In [0]:
df = spark.read.csv(
    '/Workspace/Users/vanshkrjain@gmail.com/data/Sample - Superstore.csv',
    header = True,
    inferSchema = True
)

### Basic cleaning operations

In [0]:
df = df.dropna().dropDuplicates()

In [0]:
df.count()

4996

### Storing first(initial) dataset

In [0]:
df.write.format('delta')\
    .mode('overwrite') \
    .option('delta.columnMapping.mode', 'name') \
    .saveAsTable('workspace.default.superstore_delta')

### Creating incrementla dataset

In [0]:
incremental_df = spark.read.csv(
    "/Workspace/Users/vanshkrjain@gmail.com/data/superstore_incremental.csv",
    header = True,
    inferSchema = True
    )

In [0]:
incremental_df = incremental_df.dropna().dropDuplicates()

### Performing incremental loading

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    'workspace.default.superstore_delta'
)

delta_table.alias('source_table').merge(
    incremental_df.alias('incremental_table'),
    "source_table.'Order ID' = incremental_table.'Order ID'"
 ).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Checking for incremental loading

In [0]:
incremental = delta_table.toDF().count()
print(f'Rows after merge: {incremental}')

Rows after merge: 9994


In [0]:
display(delta_table.history())

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-07-05T13:03:21.000Z,72110899362303,vanshkrjain@gmail.com,MERGE,"Map(predicate -> [""(Order ID#12755 = Order ID#12794)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2183535203898415),6bea74d5-9973-4eb9-a9fd-b38308eea866,0705-124541-7nfamwx3-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 209543, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4961, materializeSourceTimeMs -> 819, numTargetRowsInserted -> 4998, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2375, numTargetRowsUpdated -> 0, numOutputRows -> 4998, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4998, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1662)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-07-05T12:54:19.000Z,72110899362303,vanshkrjain@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.columnMapping.mode"":""name"",""delta.columnMapping.maxColumnId"":""21""}, statsOnLoad -> true)",null,List(2183535203898415),dfd11e85-0800-4c1f-9a5a-20d544a20529,0705-124541-7nfamwx3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 4996, numOutputBytes -> 205266)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


### Duplicate check

In [0]:
duplicates = delta_table.toDF().dropDuplicates().show()

+------+--------------+----------+----------+--------------+-----------+-------------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|      Customer Name|    Segment|      Country|         City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-------------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|  5007|CA-2015-169796|11-09-2015|11/14/2015|Standard Class|   Dp-13240|        Dean percer|Home Office|United States|New York City|    New York|      10035|   East|TEC-MA-10000045|     Technology|  

### Final dataset

In [0]:
delta_table.toDF().show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+------------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|           City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|       Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+------------+--------+--------+---------+
|     5|US-2015-108966|10-11-2015|10/18/2015|Standard Class|   SO-20335|   Sean O'Donnell|   Consumer|United States|Fort Lauderdale|     Florida|      33311|  South|OFF-ST-10000760|